# SAR Flood Mapping: Tile Explorer

Interactive exploration of SAR-based flood detection results.

This notebook provides two complementary views of the test dataset:

1. **Interactive map** (Cell 3b): all tiles plotted on a web map, coloured by ground truth water percentage. Click any tile for details.
2. **Tile viewer** (Cell 4): select a tile from the dropdown to display the Sentinel-1 VV image, ground truth mask, and contingency maps for each available method.

### Data
The notebook uses three local folders (configured in Cell 1):
- `s1_dir`: Sentinel-1 SAR tiles (2-band GeoTIFF, VV + VH, normalised 0–1)
- `mask_dir`: Ground truth flood masks (Copernicus EMS)
- One or more prediction directories: binary classification results from each method

### Methods
Results from any combination of the following methods can be visualised:
- **U-Net**: pre-trained deep learning model (STURM-Flood)
- **Random Forest**: classical ML classifier trained on STURM-Flood tiles
- **Threshold**: per-tile Otsu thresholding on the VV band

Leave a method's directory empty (`''`) if results are not yet available.



## Cell 1: Configuration

Set prediction directories per study area in `AREA_CONFIGS`.

All areas in `areas_to_load` (Cell 3) are loaded simultaneously, with no switching needed.

Use empty string `''` for any method not yet available.

In [ ]:
# Fix PROJ_LIB conflict (PostgreSQL/PostGIS vs Anaconda)
# MUST run before importing rasterio
import os
os.environ['PROJ_LIB'] = r'C:\Users\fatan\anaconda3\Lib\site-packages\rasterio\proj_data'

# Uncomment if packages not yet installed:
# !pip install ipywidgets rasterio folium contextily


In [ ]:
import os

# ============================================================
# AREA CONFIGS
# Set prediction directories for each study area.
# All areas in areas_to_load (Cell 3) are loaded at once.
# Use '' for methods not yet available.
# ============================================================
AREA_CONFIGS = {
    'karlstad': {
        's1_dir':    'S1_427_karlstad',
        'mask_dir':  'Floodmaps_427_karlstad',
        'otsu_dir':  'Otsu_427_karlstad',
        'rf_dir':    'RF_427_karlstad',
        'unet_dir':  'UNet_427_karlstad_0_5',
    },
    'goteborg': {
        's1_dir':    'S1_427_goteborg_final',
        'mask_dir':  'Floodmaps_427_goteborg_final',
        'otsu_dir':  'Otsu_427_goteborg_final',
        'rf_dir':    'RF_427_goteborg_final',
        'unet_dir':  'UNet_427_goteborg_final_0_5',
    },
    'kristianstad': {
        's1_dir':    'S1_427_kristianstad',
        'mask_dir':  'Floodmaps_427_kristianstad',
        'otsu_dir':  'Otsu_427_kristianstad',
        'rf_dir':    'RF_427_kristianstad',
        'unet_dir':  'UNet_427_kristianstad_0_5',
    },
}

# ============================================================
# FILE PREFIXES
# Each prediction file is expected to be named:
#   <prefix><tile_filename>
# e.g. 'Otsu_EMSR427_AOI08_10_83.tif'
# ============================================================
unet_prefix  = 'UNet_'
rf_prefix    = 'RF_'
otsu_prefix  = 'Otsu_'

print('Configuration loaded.')
for area, cfg in AREA_CONFIGS.items():
    s1_n = len([f for f in os.listdir(cfg['s1_dir']) if f.endswith('.tif')]) if os.path.isdir(cfg['s1_dir']) else -1
    status = f'{s1_n} tiles' if s1_n >= 0 else 'NOT FOUND'
    preds = [k for k in ['otsu_dir', 'rf_dir', 'unet_dir'] if cfg[k]]
    print(f'  {area:<12} {status}  predictions: {preds or ["-"]}')


## Cell 2: Imports

In [ ]:
import os
import glob
import warnings
import numpy as np
import rasterio
from rasterio.warp import transform_bounds
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import folium
import contextily as cx
from IPython.display import display
import ipywidgets as widgets

warnings.filterwarnings('ignore')
print('Imports OK.')


## Cell 3: Index tiles

Scans S1 and mask directories for all configured areas, reads each tile's geographic bounds and ground truth water percentage, and builds an in-memory index.

Edit `areas_to_load` to control which areas are loaded. Re-run this cell after changing areas.

In [ ]:
import json
def get_bounds_wgs84(tif_path):
    """Return tile bounds in WGS84 (lon_min, lat_min, lon_max, lat_max)."""
    with rasterio.open(tif_path) as src:
        crs = src.crs
        bounds = src.bounds
        if crs is None:
            raise ValueError('No CRS')
        if crs.to_epsg() == 4326:
            return (bounds.left, bounds.bottom, bounds.right, bounds.top)
        return transform_bounds(crs, 'EPSG:4326',
                                bounds.left, bounds.bottom,
                                bounds.right, bounds.top)

def compute_tile_metrics(mask_path, pred_dir, prefix):
    """Compute water detection metrics for a single tile.
    Returns a dict with water_f1, precision, recall, tp, fp, fn, tn,
    or None if the prediction file does not exist.
    """
    if not pred_dir:
        return None
    fname = os.path.basename(mask_path)
    pred_file = os.path.join(pred_dir, prefix + fname)
    if not os.path.exists(pred_file):
        return None
    with rasterio.open(mask_path) as src:
        mask = src.read(1).flatten()
    with rasterio.open(pred_file) as src:
        pred = src.read(1).flatten()
    valid = (mask != 99) & (pred != 255)
    if valid.sum() == 0:
        return None
    yt = (mask[valid] > 0).astype(int)
    yp = pred[valid].astype(int)
    tp = int(((yt == 1) & (yp == 1)).sum())
    fp = int(((yt == 0) & (yp == 1)).sum())
    fn = int(((yt == 1) & (yp == 0)).sum())
    tn = int(((yt == 0) & (yp == 0)).sum())
    pw = tp / (tp + fp) if (tp + fp) else 0.0
    rw = tp / (tp + fn) if (tp + fn) else 0.0
    fw = 2 * pw * rw / (pw + rw) if (pw + rw) else 0.0
    return {'water_f1': fw, 'precision': pw, 'recall': rw,
            'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn}

def index_area(area_name, cfg):
    """Index all tiles for one study area. Returns list of tile dicts."""
    s1_files = sorted(glob.glob(os.path.join(cfg['s1_dir'], '*.tif')))
    mask_lookup = {os.path.basename(p): p
                   for p in glob.glob(os.path.join(cfg['mask_dir'], '*.tif'))}
    tiles = []
    for s1_path in s1_files:
        fname = os.path.basename(s1_path)
        if fname not in mask_lookup:
            continue
        mask_path = mask_lookup[fname]
        try:
            bounds = get_bounds_wgs84(s1_path)
        except Exception:
            continue
        with rasterio.open(mask_path) as src:
            m = src.read(1).flatten()
        valid = m != 99
        water_pct = float((m[valid] > 0).mean() * 100) if valid.sum() > 0 else 0.0
        tiles.append({
            'tile_id':    fname,
            'area':       area_name,
            's1_path':    s1_path,
            'mask_path':  mask_path,
            'bounds':     bounds,
            'center_lat': (bounds[1] + bounds[3]) / 2,
            'center_lon': (bounds[0] + bounds[2]) / 2,
            'water_pct':  water_pct,
            'metrics':    None,  # computed on demand in Cell 4
        })
    return tiles

# ============================================================
# AREAS TO LOAD
# Add/remove areas as needed. Each area uses its AREA_CONFIGS entry.
# ============================================================
areas_to_load = ['karlstad', 'goteborg', 'kristianstad']  # start with one, add more after cache works

# ============================================================
# TILE INDEX CACHE
# First run: opens every .tif to read bounds + water_pct, then
# saves the index to a JSON file. Subsequent runs load from
# cache instantly (no rasterio.open calls). Delete the cache
# file to force a rebuild (e.g. after adding new tiles/areas).
# ============================================================
CACHE_FILE = 'tile_index_cache.json'

def _build_and_cache(areas, cache_path):
    """Build tile index incrementally, saving cache after each area.
    If the kernel crashed mid-build, already-cached areas are kept."""
    import gc
    # Load existing partial cache if present
    existing = []
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            existing = json.load(f)
    cached_areas = set(t['area'] for t in existing)

    all_tiles_raw = list(existing)  # keep already-cached areas

    for a in areas:
        if a not in AREA_CONFIGS:
            print(f'WARNING: {a} not in AREA_CONFIGS, skipping')
            continue
        if a in cached_areas:
            n = sum(1 for t in existing if t['area'] == a)
            print(f'  {a}: {n} tiles (from cache)')
            continue
        print(f'Indexing {a} (scanning .tif files)...')
        tiles = index_area(a, AREA_CONFIGS[a])
        for t in tiles:
            all_tiles_raw.append({
                'tile_id':    t['tile_id'],
                'area':       t['area'],
                's1_path':    t['s1_path'],
                'mask_path':  t['mask_path'],
                'bounds':     list(t['bounds']),
                'center_lat': t['center_lat'],
                'center_lon': t['center_lon'],
                'water_pct':  t['water_pct'],
            })
        # Save after EACH area so progress isn't lost on crash
        with open(cache_path, 'w') as f:
            json.dump(all_tiles_raw, f)
        print(f'  {len(tiles)} tiles (cached)')
        gc.collect()

    # Convert to tile_index format
    result = []
    for t in all_tiles_raw:
        if t['area'] not in areas:
            continue
        t['bounds'] = tuple(t['bounds']) if isinstance(t['bounds'], list) else t['bounds']
        t['metrics'] = None
        result.append(t)
    print(f'\nCache saved to {cache_path}')
    return result

def _load_cache(cache_path, areas):
    """Load tile index from JSON cache.
    Returns tiles if ALL requested areas are cached, else None."""
    if not os.path.exists(cache_path):
        return None
    with open(cache_path) as f:
        cache_data = json.load(f)
    cached_areas = set(t['area'] for t in cache_data)
    if not set(areas).issubset(cached_areas):
        missing = set(areas) - cached_areas
        print(f'Cache missing areas {missing}, will index them...')
        return None
    tiles = []
    for t in cache_data:
        if t['area'] not in areas:
            continue
        t['bounds'] = tuple(t['bounds'])
        t['metrics'] = None  # always recomputed on demand
        tiles.append(t)
    return tiles

# Try cache first, rebuild if needed
tile_index = _load_cache(CACHE_FILE, areas_to_load)
if tile_index is None:
    tile_index = _build_and_cache(areas_to_load, CACHE_FILE)
else:
    print(f'Loaded {len(tile_index)} tiles from cache')

print(f'\nTotal indexed:  {len(tile_index)} tiles')
print(f'With water >1%: {sum(1 for t in tile_index if t["water_pct"] > 1)}')
for a in areas_to_load:
    n = sum(1 for t in tile_index if t['area'] == a)
    print(f'  {a}: {n} tiles')


## Cell 3b: Interactive map

Tiles are coloured by ground truth water percentage.
Click any tile rectangle to see its ID and water content.



In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# Continuous Blues colormap matching the static map
_max_pct = max(t['water_pct'] for t in tile_index)
_norm    = mcolors.Normalize(vmin=0, vmax=_max_pct)
_cmap    = cm.get_cmap('Blues')


def water_pct_color(pct):
    """Map ground truth water percentage to a hex colour (Blues colormap)."""
    if pct <= 0:
        return '#cccccc'  # grey for tiles with no water
    return mcolors.to_hex(_cmap(_norm(pct)))

def make_popup(tile):
    wp = round(tile['water_pct'], 1)
    html = (f"<div style='font-size:11px;min-width:260px'>"
            f"<b>{tile['tile_id']}</b><br>"
            f"Ground truth water: {wp}%<br><br>"
            f"<i>Select tile in the viewer below for full metrics and imagery.</i>"
            f"</div>")
    return folium.Popup(html, max_width=320)

# Build interactive folium map centred on the tile extent
lats = [t['center_lat'] for t in tile_index]
lons = [t['center_lon'] for t in tile_index]
fmap = folium.Map(
    location=[np.mean(lats), np.mean(lons)],
    zoom_start=8,
    tiles='OpenStreetMap',
    control_scale=True
)

# Add one rectangle per tile
for tile in tile_index:
    color = water_pct_color(tile['water_pct'])
    b = tile['bounds']
    wp = round(tile['water_pct'], 1)
    folium.Rectangle(
        bounds=[[b[1], b[0]], [b[3], b[2]]],
        color=color, fill=True, fill_color=color,
        fill_opacity=0.5, weight=1,
        popup=make_popup(tile),
        tooltip=f"{tile['tile_id']} | Water GT: {wp}%"
    ).add_to(fmap)

# Legend with gradient bar 
legend = f"""
<div style='position:fixed;top:30px;right:30px;z-index:1000;
     background:white;padding:10px;border:1px solid grey;font-size:12px;'>
  <b>Ground truth water %</b><br>
  <div style='background:linear-gradient(to right,#deebf7,#08306b);
   width:120px;height:12px;display:inline-block;'></div><br>
  <span style='float:left;font-size:10px;'>0%</span>
  <span style='float:right;font-size:10px;'>{_max_pct:.0f}%</span>
  <br><br>
  <span style='color:#ccc'>&#9632;</span> no water
</div>"""
fmap.get_root().html.add_child(folium.Element(legend))

fmap

## Cell 4: Tile viewer

Select a tile from the dropdown and click **Show tile** to display:
- Sentinel-1 VV backscatter (normalised 0–1)
- Ground truth flood mask
- Contingency map for each available prediction method

Each new selection is appended below. Scroll down to compare tiles.

Metrics (Precision, Recall, F1) are computed on demand from the prediction GeoTIFFs.

In [ ]:
def make_contingency(yt, yp):
    """Create contingency map: 0=TN, 1=TP, 2=FP, 3=FN."""
    cont = np.zeros_like(yt)
    cont[(yt == 1) & (yp == 1)] = 1
    cont[(yt == 0) & (yp == 1)] = 2
    cont[(yt == 1) & (yp == 0)] = 3
    return cont

def compute_tile_metrics(mask_path, pred_dir, prefix):
    """Compute water F1, precision, recall for a single tile on demand."""
    if not pred_dir:
        return None
    fname = os.path.basename(mask_path)
    pred_file = os.path.join(pred_dir, prefix + fname)
    if not os.path.exists(pred_file):
        return None
    with rasterio.open(mask_path) as src:
        mask = src.read(1).flatten()
    with rasterio.open(pred_file) as src:
        pred = src.read(1).flatten()
    valid = (mask != 99) & (pred != 255)
    if valid.sum() == 0:
        return None
    yt = (mask[valid] > 0).astype(int)
    yp = pred[valid].astype(int)
    tp = int(((yt == 1) & (yp == 1)).sum())
    fp = int(((yt == 0) & (yp == 1)).sum())
    fn = int(((yt == 1) & (yp == 0)).sum())
    tn = int(((yt == 0) & (yp == 0)).sum())
    pw = tp / (tp + fp) if (tp + fp) else 0.0
    rw = tp / (tp + fn) if (tp + fn) else 0.0
    fw = 2 * pw * rw / (pw + rw) if (pw + rw) else 0.0
    return {'water_f1': fw, 'precision': pw, 'recall': rw,
            'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn}

In [ ]:
# ============================================================
# DISPLAY CONFIG: toggle panels and adjust styling
# ============================================================
SHOW_VV         = True
SHOW_VH         = True    # Toggle Sentinel-1 VH panel
SHOW_ORTHO      = True
SHOW_GT         = True
SHOW_PREDS      = True     # Show U-Net, RF, Otsu contingency maps
METHODS_TO_SHOW = ['U-Net', 'Threshold (Otsu)']   # choose from: 'U-Net', 'RF', 'Threshold (Otsu)'`


SHOW_SUPTITLE   = False     # Tile ID + GT water% above figure
SHOW_PANEL_TITLES = True   # Titles above each subplot (S1 VV, GT, etc.)
SHOW_METRICS_IN_TITLES = False  # F1/P/R below method name in prediction panels
SHOW_LEGEND     = True     # TP/FP/FN/TN legend below figure

FONT_SUPTITLE   = 20       # Suptitle (tile ID + water%)
FONT_PANEL      = 20       # Panel titles (S1 VV, Ortophoto, etc.)
FONT_LEGEND     = 20       # Legend text


def visualize_tile(tile_id):
    """Display S1 bands, ortophoto, ground truth, and contingency maps."""
    tile = next((t for t in tile_index if t['tile_id'] == tile_id), None)
    if not tile:
        print(f'Tile not found: {tile_id}')
        return

    # Load radar bands and ground truth
    with rasterio.open(tile['s1_path']) as src:
        vv = src.read(1)
        vh = src.read(2) if SHOW_VH else None
        bounds = src.bounds
        tile_crs = src.crs
    with rasterio.open(tile['mask_path']) as src:
        mask = src.read(1)
    yt = np.where(mask == 99, -1, (mask > 0).astype(np.int8))

    # Get prediction directories for this area
    area_cfg = AREA_CONFIGS.get(tile.get('area', ''), {})
    _unet_dir = area_cfg.get('unet_dir', '')
    _rf_dir   = area_cfg.get('rf_dir', '')
    _otsu_dir = area_cfg.get('otsu_dir', '')

    # Load predictions for each method (only if showing them)
    methods = [
        ('U-Net',            _unet_dir, unet_prefix, 'unet'),
        ('RF',               _rf_dir,   rf_prefix,   'rf'),
        ('Threshold (Otsu)', _otsu_dir, otsu_prefix, 'otsu'),
    ]
    methods = [m for m in methods if m[0] in METHODS_TO_SHOW]
    preds = {}
    if SHOW_PREDS:
        for name, pred_dir, prefix, key in methods:
            pred_file = os.path.join(pred_dir, prefix + tile_id) if pred_dir else ''
            if pred_dir and os.path.exists(pred_file):
                with rasterio.open(pred_file) as src:
                    p = src.read(1)
                preds[name] = np.where(p == 255, -1, p.astype(np.int8))
            else:
                preds[name] = None
        available = [m for m in methods if preds[m[0]] is not None]
    else:
        available = []

    # Compute number of panels based on toggles
    n_base = sum([SHOW_VV, SHOW_VH, SHOW_ORTHO, SHOW_GT])
    ncols = n_base + len(available)
    if ncols == 0:
        print('No panels to show. Set at least one SHOW_* flag to True.')
        return

    fig, axes = plt.subplots(1, ncols, figsize=(4 * ncols, 4.5), dpi=150)
    if ncols == 1:
        axes = [axes]  # Make iterable when only one panel

    # Suptitle
    if SHOW_SUPTITLE:
        wp = round(tile['water_pct'], 1)
        fig.suptitle(f'{tile_id}   |   Ground truth water: {wp}%',
                     fontsize=FONT_SUPTITLE, fontweight='bold', y=1.02)

    col = 0  # Current panel index

    # Sentinel-1 VV
    if SHOW_VV:
        axes[col].imshow(vv, cmap='gray', vmin=0, vmax=1)
        if SHOW_PANEL_TITLES:
            axes[col].set_title('S1 VV', fontsize=FONT_PANEL)
        axes[col].axis('off')
        col += 1

    # Sentinel-1 VH
    if SHOW_VH:
        axes[col].imshow(vh, cmap='gray', vmin=0, vmax=1)
        if SHOW_PANEL_TITLES:
            axes[col].set_title('S1 VH', fontsize=FONT_PANEL)
        axes[col].axis('off')
        col += 1

    # Ortophoto (ESRI World Imagery)
    if SHOW_ORTHO:
        try:
            import contextily as cx
            axes[col].set_xlim(bounds.left, bounds.right)
            axes[col].set_ylim(bounds.bottom, bounds.top)
            cx.add_basemap(axes[col], crs=tile_crs,
                           source=cx.providers.Esri.WorldImagery,
                           attribution=False)
            axes[col].set_aspect('equal')
            if SHOW_PANEL_TITLES:
                axes[col].set_title('Ortophoto', fontsize=FONT_PANEL)
        except Exception as e:
            print(f"Basemap error: {e}")
            axes[col].text(0.5, 0.5, 'No basemap', ha='center', va='center',
                           transform=axes[col].transAxes)
            if SHOW_PANEL_TITLES:
                axes[col].set_title('Ortophoto (offline)', fontsize=FONT_PANEL)
        axes[col].axis('off')
        col += 1

    # Ground truth
    if SHOW_GT:
        gt_cmap = ListedColormap(['#F3F3F3', '#1C5781'])
        axes[col].imshow(np.where(yt < 0, 0, yt), cmap=gt_cmap, vmin=0, vmax=1)
        if SHOW_PANEL_TITLES:
            axes[col].set_title('Ground Truth', fontsize=FONT_PANEL)
        axes[col].axis('off')
        col += 1

    # Contingency maps per method
    if SHOW_PREDS:
        cont_cmap = ListedColormap(['#F3F3F3', '#1C5781', '#8F1834', '#F7BB65'])
        for (name, _, _, key) in available:
            yp = preds[name]
            valid = (yt >= 0) & (yp >= 0)
            cont = make_contingency(
                np.where(valid, yt, 0),
                np.where(valid, yp, 0)
            )
            axes[col].imshow(cont, cmap=cont_cmap, vmin=0, vmax=3)

            if SHOW_PANEL_TITLES:
                if SHOW_METRICS_IN_TITLES:
                    met = compute_tile_metrics(
                        tile['mask_path'],
                        [_unet_dir, _rf_dir, _otsu_dir][['unet','rf','otsu'].index(key)],
                        [unet_prefix, rf_prefix, otsu_prefix][['unet','rf','otsu'].index(key)]
                    )
                    title = (f"{name}\n"
                             f"F1={met['water_f1']:.3f}  P={met['precision']:.3f}  R={met['recall']:.3f}") if met else name
                else:
                    title = name
                axes[col].set_title(title, fontsize=FONT_PANEL)
            axes[col].axis('off')
            col += 1

    # Shared legend
    if SHOW_LEGEND and (SHOW_GT or SHOW_PREDS):
        patches = [
            mpatches.Patch(color='#1C5781', label='TP — correctly detected water'),
            mpatches.Patch(color='#8F1834', label='FP — water predicted, none present'),
            mpatches.Patch(color='#F7BB65', label='FN — water missed'),
            mpatches.Patch(color='#F3F3F3', label='TN — correctly detected non-water'),
        ]
        fig.legend(handles=patches, loc='lower center', ncol=4,
                   fontsize=FONT_LEGEND, bbox_to_anchor=(0.5, -0.04))

    plt.tight_layout()
    plt.show()


In [ ]:
def _draw_tile_panels(axes_row, tile_id, show_panel_titles=True):
    """Draw all panels for one tile into the given row of axes."""
    tile = next((t for t in tile_index if t['tile_id'] == tile_id), None)
    if not tile:
        print(f'Tile not found: {tile_id}')
        return

    with rasterio.open(tile['s1_path']) as src:
        vv = src.read(1)
        vh = src.read(2) if SHOW_VH else None
        bounds = src.bounds
        tile_crs = src.crs
    with rasterio.open(tile['mask_path']) as src:
        mask = src.read(1)
    yt = np.where(mask == 99, -1, (mask > 0).astype(np.int8))

    area_cfg = AREA_CONFIGS.get(tile.get('area', ''), {})
    _unet_dir = area_cfg.get('unet_dir', '')
    _rf_dir   = area_cfg.get('rf_dir', '')
    _otsu_dir = area_cfg.get('otsu_dir', '')

    methods = [
        ('U-Net',            _unet_dir, unet_prefix, 'unet'),
        ('RF',               _rf_dir,   rf_prefix,   'rf'),
        ('Threshold (Otsu)', _otsu_dir, otsu_prefix, 'otsu'),
    ]
    methods = [m for m in methods if m[0] in METHODS_TO_SHOW]
    preds = {}
    if SHOW_PREDS:
        for name, pred_dir, prefix, key in methods:
            pred_file = os.path.join(pred_dir, prefix + tile_id) if pred_dir else ''
            if pred_dir and os.path.exists(pred_file):
                with rasterio.open(pred_file) as src:
                    p = src.read(1)
                preds[name] = np.where(p == 255, -1, p.astype(np.int8))
            else:
                preds[name] = None
        available = [m for m in methods if preds[m[0]] is not None]
    else:
        available = []

    col = 0

    if SHOW_VV:
        axes_row[col].imshow(vv, cmap='gray', vmin=0, vmax=1)
        if show_panel_titles:
            axes_row[col].set_title('S1 VV', fontsize=FONT_PANEL)
        axes_row[col].axis('off'); col += 1

    if SHOW_VH:
        axes_row[col].imshow(vh, cmap='gray', vmin=0, vmax=1)
        if show_panel_titles:
            axes_row[col].set_title('S1 VH', fontsize=FONT_PANEL)
        axes_row[col].axis('off'); col += 1

    if SHOW_ORTHO:
        try:
            import contextily as cx
            axes_row[col].set_xlim(bounds.left, bounds.right)
            axes_row[col].set_ylim(bounds.bottom, bounds.top)
            cx.add_basemap(axes_row[col], crs=tile_crs,
                           source=cx.providers.Esri.WorldImagery,
                           attribution=False)
            axes_row[col].set_aspect('equal')
            if show_panel_titles:
                axes_row[col].set_title('Ortophoto', fontsize=FONT_PANEL)
        except Exception as e:
            print(f"Basemap error: {e}")
        axes_row[col].axis('off'); col += 1

    if SHOW_GT:
        gt_cmap = ListedColormap(['#F3F3F3', '#1C5781'])
        axes_row[col].imshow(np.where(yt < 0, 0, yt), cmap=gt_cmap, vmin=0, vmax=1)
        if show_panel_titles:
            axes_row[col].set_title('Ground Truth', fontsize=FONT_PANEL)
        axes_row[col].axis('off'); col += 1

    if SHOW_PREDS:
        cont_cmap = ListedColormap(['#F3F3F3', '#1C5781', '#8F1834', '#F7BB65'])
        for (name, _, _, key) in available:
            yp = preds[name]
            valid = (yt >= 0) & (yp >= 0)
            cont = make_contingency(np.where(valid, yt, 0),
                                    np.where(valid, yp, 0))
            axes_row[col].imshow(cont, cmap=cont_cmap, vmin=0, vmax=3)
            if show_panel_titles:
                axes_row[col].set_title(name, fontsize=FONT_PANEL)
            axes_row[col].axis('off'); col += 1


def visualize_tiles_grid(tile_ids, row_labels=None, output_path=None, dpi=300):
    """Composite figure with multiple tiles as rows and a shared legend at the bottom."""
    n_rows = len(tile_ids)

    n_base = sum([SHOW_VV, SHOW_VH, SHOW_ORTHO, SHOW_GT])
    n_pred_cols = len(METHODS_TO_SHOW) if SHOW_PREDS else 0
    ncols = n_base + n_pred_cols

    fig, axes = plt.subplots(n_rows, ncols,
                             figsize=(4 * ncols, 4.5 * n_rows), dpi=150)
    if n_rows == 1:
        axes = axes.reshape(1, -1)

    for row_idx, tile_id in enumerate(tile_ids):
        _draw_tile_panels(axes[row_idx], tile_id, show_panel_titles=(row_idx == 0))

    plt.tight_layout()

    if row_labels:
        fig.canvas.draw()
        for row_idx, label in enumerate(row_labels):
            pos = axes[row_idx, 0].get_position()
            fig.text(pos.x0 - 0.005, (pos.y0 + pos.y1) / 2, label,
                     ha='right', va='center', fontsize=14, fontweight='bold')

    if SHOW_LEGEND:
        patches = [
            mpatches.Patch(color='#1C5781', label='TP — correctly detected water'),
            mpatches.Patch(color='#8F1834', label='FP — water predicted, none present'),
            mpatches.Patch(color='#F7BB65', label='FN — water missed'),
            mpatches.Patch(color='#F3F3F3', label='TN — correctly detected non-water'),
        ]
        fig.legend(handles=patches, loc='lower center', ncol=4,
                   fontsize=FONT_LEGEND, bbox_to_anchor=(0.5, -0.1))

    if output_path:
        plt.savefig(output_path, dpi=dpi, bbox_inches='tight')
        print(f"Sparat -> {output_path}")

    plt.show()

In [ ]:
tile_ids = [t['tile_id'] for t in tile_index]

dd = widgets.Dropdown(options=tile_ids, description='Tile:',
                      layout=widgets.Layout(width='70%'))
txt = widgets.Text(placeholder='Paste tile ID from map popup here, then press Enter',
                   description='Quick jump:',
                   style={'description_width': 'initial'},
                   layout=widgets.Layout(width='70%'))
btn = widgets.Button(description='Show tile', button_style='primary')
out = widgets.Output()

def on_click(b):
    with out:
        visualize_tile(dd.value)

def on_text_submit(change):
    tid = txt.value.strip()
    if tid in tile_ids:
        dd.value = tid
        with out:
            visualize_tile(tid)
    elif tid:
        with out:
            print(f'Tile not found: {tid}')
    txt.value = ''

btn.on_click(on_click)
txt.on_submit(on_text_submit)
display(txt, dd, btn, out)

In [ ]:
%pip list | findstr -i widget

## Clear tile index cache

Run this cell to delete the cached tile index (`tile_index_cache.json`).
The cache speeds up subsequent runs by skipping filesystem indexing,
but it stores absolute file paths, so the cache must be cleared after:

- Updating `AREA_CONFIGS` (renamed or replaced directories)
- Re-running data preparation with updated ground truth
- Adding or removing study areas

After clearing, re-run **Cell 3** to rebuild the index from scratch.

In [ ]:
import os

cache_path = 'tile_index_cache.json'
if os.path.exists(cache_path):
    os.remove(cache_path)
    print(f"✓ Raderade {cache_path}")
else:
    print("Cache finns inte")

## Final .png output

Combines multiple tiles into a single PNG figure using `visualize_tiles_grid`. Each row corresponds to one tile and includes the configured panels (Sentinel-1 bands, ortophoto, ground truth, and contingency maps per method).

Use `tile_ids` to select which tiles to include and optional `row_labels` to label each row. The figure is saved to `output_path`.


In [ ]:
visualize_tiles_grid(
    tile_ids=[
        'EMSR427_AOI07_14_39.tif',   # your choice
        #'EMSR427_AOI04_3_48.tif',   # Göteborg, your choice
        #'EMSR427_AOI07_28_59.tif',   # Kristianstad, your choice
    ],
    #row_labels=['Karlstad', 'Göteborg', 'Kristianstad'],
    output_path='figur_X_cross_method.png',
)

In [ ]:
!pip install matplotlib-scalebar
!pip install shapely

## Overview maps

Mosaics tile predictions and tile ground truth into whole-area rasters and plots confusion maps (TP/FP/FN colour-coded, TN transparent) over an ESRI World Imagery basemap, with a scalebar and north arrow.

Uses the same hex colours as the tile view for visual consistency.

Requires: `pip install matplotlib-scalebar`

In [ ]:
# ============================================================
# OVERVIEW MAPS: CONFIGURATION
# Toggle areas and methods via the lists below.
# Output is saved as <OVERVIEW_OUTPUT_DIR>/oversikt_<area>_<method>.png
# ============================================================
OVERVIEW_OUTPUT_DIR     = 'oversiktskartor'
OVERVIEW_DPI            = 150
OVERVIEW_BASEMAP_ZOOM   = 11    # contextily zoom; higher = sharper but slower (try 13-15)
OVERVIEW_DRAW_BOUNDARY  = True  # draws an outline around the actual data footprint
OVERVIEW_BOUNDARY_COLOR = '#888888'
OVERVIEW_BOUNDARY_WIDTH = 1.0
OVERVIEW_PADDING_FRACTION = 0.03    # margin around data (3% of map width)

# Place-name labels (Karlstad, Göteborg, etc.)
OVERVIEW_SHOW_PLACE_LABELS = True
OVERVIEW_LABEL_FONTSIZE    = 10
OVERVIEW_LABEL_MARKERSIZE  = 6

# Toggle which methods to generate
METHODS_FOR_OVERVIEW = ['unet', 'rf', 'otsu']    # choose from: 'unet', 'rf', 'otsu'

# Toggle which areas to generate
AREAS_FOR_OVERVIEW   = ['karlstad']

os.makedirs(OVERVIEW_OUTPUT_DIR, exist_ok=True)
print(f"Output: {OVERVIEW_OUTPUT_DIR}/")
print(f"Combinations to generate: "
      f"{len(AREAS_FOR_OVERVIEW)} areas × {len(METHODS_FOR_OVERVIEW)} methods "
      f"= {len(AREAS_FOR_OVERVIEW)*len(METHODS_FOR_OVERVIEW)} maps")


In [ ]:
# ============================================================
# OVERVIEW MAPS: HELPER FUNCTIONS AND MAIN FUNCTION
# ============================================================
from rasterio.merge import merge as rio_merge
from rasterio.plot import plotting_extent
from rasterio.features import shapes as rio_shapes
from matplotlib.patches import Polygon as _MplPolygon
from matplotlib.lines import Line2D as _Line2D
import matplotlib.patheffects as _pe
from rasterio.warp import transform as _rio_transform

# matplotlib-scalebar for the scalebar
try:
    from matplotlib_scalebar.scalebar import ScaleBar
    _HAVE_SCALEBAR = True
except ImportError:
    print("WARNING: matplotlib-scalebar missing. Install: pip install matplotlib-scalebar")
    _HAVE_SCALEBAR = False


# Mapping: method key -> (config key, prefix, display name)
_METHOD_DIR_MAP = {
    'unet': ('unet_dir', unet_prefix, 'U-Net'),
    'rf':   ('rf_dir',   rf_prefix,   'Random Forest'),
    'otsu': ('otsu_dir', otsu_prefix, 'Otsu'),
}

# Mapping: area key -> (display name, AOI code) for titles
_AREA_DISPLAY_MAP = {
    'karlstad':     ('Karlstad',     'AOI08'),
    'goteborg':     ('Göteborg',     'AOI04'),
    'kristianstad': ('Kristianstad', 'AOI07'),
}

# Mapping: area key -> list of (name, lon, lat) in WGS84 for place labels.
# Places outside the map extent are silently skipped.
_AREA_PLACES = {
    'karlstad': [
        ('Karlstad',    13.5036, 59.3793),
        ('Grums',       13.1100, 59.3508),
        ('Säffle',      12.9264, 59.1336),
        ('Åmål',        12.7050, 59.0517),
        ('Vänersborg',  12.3236, 58.3811),
    ],
    'goteborg': [
        ('Göteborg',    11.9746, 57.7089),
        ('Mölndal',     12.0136, 57.6554),
        ('Trollhättan', 12.2886, 58.2837),
        ('Kungälv',     11.9750, 57.8704),
        ('Alingsås',    12.5333, 57.9300),
    ],
    'kristianstad': [
        ('Kristianstad', 14.1567, 56.0294),
        ('Hässleholm',   13.7660, 56.1592),
        ('Åhus',         14.3097, 55.9286),
        ('Osby',         13.9911, 56.3811),
        ('Vinslöv',      13.9119, 56.0508),
    ],
}


def _hex_to_rgba(hex_color, alpha=1.0):
    """Convert hex string (#RRGGBB) to [r, g, b, alpha] (0-1)."""
    h = hex_color.lstrip('#')
    return [int(h[i:i+2], 16) / 255.0 for i in (0, 2, 4)] + [alpha]


def _matched_tile_paths(area, method):
    """Find tiles where both GT and prediction exist.
    Returns list [(mask_path, pred_path), ...]
    """
    cfg = AREA_CONFIGS[area]
    dir_key, prefix, _ = _METHOD_DIR_MAP[method]
    pred_dir = cfg.get(dir_key, '')
    mask_dir = cfg.get('mask_dir', '')

    if not pred_dir or not os.path.isdir(pred_dir):
        return []
    if not mask_dir or not os.path.isdir(mask_dir):
        return []

    pairs = []
    for mask_file in sorted(glob.glob(os.path.join(mask_dir, '*.tif'))):
        fname = os.path.basename(mask_file)
        pred_file = os.path.join(pred_dir, prefix + fname)
        if os.path.exists(pred_file):
            pairs.append((mask_file, pred_file))
    return pairs


def _mosaic_files(file_paths, nodata=None):
    """Mosaic a list of raster files. Returns (data, transform, crs).
    nodata: fill value for areas outside all input tiles.
    """
    srcs = [rasterio.open(p) for p in file_paths]
    try:
        if nodata is not None:
            mosaic, transform = rio_merge(srcs, nodata=nodata)
        else:
            mosaic, transform = rio_merge(srcs)
        crs = srcs[0].crs
    finally:
        for s in srcs:
            s.close()
    return mosaic[0], transform, crs


def _draw_place_labels(ax, area, gt_crs, extent):
    """Plot place markers and labels with a white halo (cartographic style).
    Coordinates in _AREA_PLACES are WGS84 and transformed to the map's CRS.
    Places outside the visible extent are silently skipped.
    """
    places = _AREA_PLACES.get(area, [])
    if not places:
        return

    lons = [p[1] for p in places]
    lats = [p[2] for p in places]
    xs, ys = _rio_transform('EPSG:4326', gt_crs, lons, lats)

    for (name, _, _), x, y in zip(places, xs, ys):
        if not (extent[0] <= x <= extent[1] and extent[2] <= y <= extent[3]):
            continue
        ax.plot(x, y, 'o',
                color='white', markeredgecolor='black',
                markersize=OVERVIEW_LABEL_MARKERSIZE,
                zorder=15)
        ax.annotate(name, xy=(x, y), xytext=(7, 5),
                    textcoords='offset points',
                    fontsize=OVERVIEW_LABEL_FONTSIZE,
                    fontweight='bold', color='black',
                    path_effects=[_pe.withStroke(linewidth=2.5,
                                                  foreground='white')],
                    zorder=15)


def make_overview_map(area, method, save_path=None, dpi=300):
    """Produce an overview map for a combination of area and method.

    Mosaics GT and prediction tiles, computes TP/FP/FN per pixel,
    plots with transparent TN, ESRI World Imagery basemap, scalebar
    and north arrow. Saves to save_path if provided.
    """
    method_low = method.lower()
    if method_low not in _METHOD_DIR_MAP:
        print(f"  Unknown method: {method}")
        return

    # Build the figure title (Swedish label, kept for the thesis figures)
    area_display, aoi_code = _AREA_DISPLAY_MAP.get(area, (area.capitalize(), ''))
    _, _, method_display = _METHOD_DIR_MAP[method_low]
    title_text = f"{method_display}: konfusionskarta över {area_display} ({aoi_code})"

    pairs = _matched_tile_paths(area, method_low)
    if not pairs:
        print(f"  {area}/{method}: no matching tiles, skipping")
        return

    mask_files = [p[0] for p in pairs]
    pred_files = [p[1] for p in pairs]

    print(f"  {area}/{method}: mosaicking {len(pairs)} tiles ...", end='', flush=True)

    gt,   gt_tf,   gt_crs   = _mosaic_files(mask_files, nodata=99)
    pred, pred_tf, pred_crs = _mosaic_files(pred_files, nodata=255)

    # Align shape in case mosaics have slightly different dimensions
    h = min(gt.shape[0], pred.shape[0])
    w = min(gt.shape[1], pred.shape[1])
    gt   = gt[:h, :w]
    pred = pred[:h, :w]

    # Compute confusion classes
    valid = (gt != 99) & (pred != 255)
    yt    = (gt > 0).astype(np.int8)
    yp    = pred.astype(np.int8)

    tp = valid & (yt == 1) & (yp == 1)
    fp = valid & (yt == 0) & (yp == 1)
    fn = valid & (yt == 1) & (yp == 0)

    # RGBA overlay (TN fully transparent by default)
    rgba = np.zeros((h, w, 4), dtype=np.float32)
    rgba[tp] = _hex_to_rgba('#1C5781')   # TP: blue
    rgba[fp] = _hex_to_rgba('#8F1834')   # FP: red
    rgba[fn] = _hex_to_rgba('#F7BB65')   # FN: orange

    print(f" plotting ...", end='', flush=True)

    # Plot
    fig, ax = plt.subplots(figsize=(12, 10))
    extent = plotting_extent(gt, gt_tf)

    # Compute padding for margin around data
    xpad = (extent[1] - extent[0]) * OVERVIEW_PADDING_FRACTION
    ypad = (extent[3] - extent[2]) * OVERVIEW_PADDING_FRACTION

    ax.imshow(rgba, extent=extent, origin='upper', interpolation='nearest', zorder=10)
    ax.set_xlim(extent[0] - xpad, extent[1] + xpad)
    ax.set_ylim(extent[2] - ypad, extent[3] + ypad)

    # Basemap (ESRI World Imagery, same as in tile view)
    try:
        cx.add_basemap(ax, crs=gt_crs, source=cx.providers.Esri.WorldImagery,
                       attribution=False, zoom=OVERVIEW_BASEMAP_ZOOM)
    except Exception as e:
        print(f"\n  Basemap error: {e}")

    # Restore extent with padding (basemap may have changed it)
    ax.set_xlim(extent[0] - xpad, extent[1] + xpad)
    ax.set_ylim(extent[2] - ypad, extent[3] + ypad)

    # Scalebar (UTM 33N in metres)
    if _HAVE_SCALEBAR:
        ax.add_artist(ScaleBar(
            dx=1, units='m',
            location='lower right',
            length_fraction=0.20,
            box_alpha=0.85,
            color='black',
            font_properties={'size': 7},
            border_pad=1.1             # matches the legend's borderaxespad
        ))

    # North arrow
    ax.annotate('', xy=(0.95, 0.95), xytext=(0.95, 0.90),
                xycoords=ax.transAxes,
                arrowprops=dict(facecolor='#D6D8D3', edgecolor='black',
                                width=4, headwidth=12))
    #ax.text(0.95, 0.96, 'N', transform=ax.transAxes,
            #ha='center', va='bottom', fontsize=12, fontweight='bold')

    # Title block (upper-left corner, classic cartographic layout)
    ax.text(0.03, 0.97, title_text,
            transform=ax.transAxes,
            ha='left', va='top',
            fontsize=11, fontweight='bold',
            bbox=dict(boxstyle='square,pad=0.5',
                      facecolor='white',
                      edgecolor='black',
                      linewidth=0.8,
                      alpha=0.92),
            zorder=21)

    # Place-name labels (zorder=15 so legend/title block in corners stay on top)
    if OVERVIEW_SHOW_PLACE_LABELS:
        _draw_place_labels(ax, area, gt_crs, extent)

    # Legend
    legend_handles = [
        mpatches.Patch(color='#1C5781', label='TP — korrekt detekterat vatten'),
        mpatches.Patch(color='#8F1834', label='FP — falsk prediktion'),
        mpatches.Patch(color='#F7BB65', label='FN — missat vatten'),
    ]
    if OVERVIEW_DRAW_BOUNDARY:
        legend_handles.append(
            _Line2D([0], [0],
                    color=OVERVIEW_BOUNDARY_COLOR,
                    linewidth=OVERVIEW_BOUNDARY_WIDTH,
                    label='Studieområdets gräns')
        )

    leg = ax.legend(
        handles=legend_handles,
        loc='lower right',
        framealpha=0.9,
        fontsize=8,          # bumped (was 5)
        handlelength=1.2,     # bumped (was 1.0)
        handletextpad=0.7,
        borderpad=0.5,        # bumped (was 0.5)
        labelspacing=0.5,     # bumped (was 0.5)
        #borderaxespad=6.5,    # distance from axis edge (default 0.5)
        bbox_to_anchor=(0.99, 0.15),    # replaces borderaxespad
    )
    leg.set_zorder(20)        # above everything (boundary is 12, overlay is 10)

    # Boundary around actual data footprint (where gt != 99)
    if OVERVIEW_DRAW_BOUNDARY:
        valid_for_outline = (gt != 99).astype('uint8')
        shape_gen = rio_shapes(valid_for_outline, mask=(valid_for_outline > 0),
                                transform=gt_tf)
        for geom, val in shape_gen:
            if val != 1 or geom.get('type') != 'Polygon':
                continue
            for ring_idx, ring in enumerate(geom['coordinates']):
                if len(ring) < 4:
                    continue
                style = '-' if ring_idx == 0 else '--'   # outer = solid, holes = dashed
                ax.add_patch(_MplPolygon(
                    ring, closed=True, fill=False,
                    edgecolor=OVERVIEW_BOUNDARY_COLOR,
                    linewidth=OVERVIEW_BOUNDARY_WIDTH,
                    linestyle=style,
                    zorder=12
                ))

    ax.set_xticks([])
    ax.set_yticks([])

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=dpi, bbox_inches='tight')
        print(f" saved -> {save_path}")
    else:
        print(" done")

    plt.show()
    plt.close(fig)


In [ ]:
# ============================================================
# OVERVIEW MAPS: GENERATE
# Loop over selected areas × methods. Each map is saved as PNG.
# ============================================================
print(f"Generating overview maps in '{OVERVIEW_OUTPUT_DIR}/' ...")
print()

for area in AREAS_FOR_OVERVIEW:
    for method in METHODS_FOR_OVERVIEW:
        save_path = os.path.join(
            OVERVIEW_OUTPUT_DIR,
            f"oversikt_{area}_{method}.png"
        )
        make_overview_map(area, method,
                          save_path=save_path,
                          dpi=OVERVIEW_DPI)

print()
print("Done.")
